In [41]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_openai.llms import OpenAI
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI 
from langchain.messages import HumanMessage


In [17]:
load_dotenv(override=True)

True

In [18]:
loader = PyPDFLoader("Ramatoulaye_Diawane_CV-Pro.pdf")

In [19]:
tokennizer = tiktoken.encoding_for_model("gpt-4o-mini")

In [20]:
print(tokennizer.name)

o200k_base


In [28]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=tokennizer.name,
    chunk_size=300,
    chunk_overlap=2,
    )

In [29]:
chunks = loader.load_and_split(splitter)

In [30]:
print(len(chunks))

3


In [24]:
print(chunks[0].metadata)

{'producer': '', 'creator': 'WPS Writer', 'creationdate': '2026-03-28T11:10:36+01:00', 'author': '', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-03-28T11:10:36+01:00', 'sourcemodified': "D:20260328111036+01'00'", 'subject': '', 'title': 'Ramatoulaye DIAWANE', 'trapped': '/False', 'source': 'Ramatoulaye_Diawane_CV-Pro.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


In [31]:
embeddings_model = OpenAIEmbeddings()

In [32]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="CV_data_collection"
)

In [33]:
retriever = vector_store.as_retriever(kwargs= {"k":10})

In [36]:
## Searching information about cantidates in the resume
@tool
def retriever_tool(query : str) -> str:
     """
     Permet de chercher des informations sur des candidats:
     -Nom, Prénom, Diplômes
     -Expériences professionnelles
     -Compétences techniques
     """
     relevent_chunks=retriever.invoke(query)
     context_list = [d.page_content for d in relevent_chunks]
     context = ".".joint(context_list)
     return context



In [47]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt="Réponds à la question de l'utilisateur en utilisant les tools fournies. "

)

In [48]:
resp = agent.invoke(input={
    "messages":[
        HumanMessage("Nom, prénom, diplômes") 
    ]
})

AttributeError: 'str' object has no attribute 'joint'

In [43]:
print(resp['messages'][-1].content)

Je ne sais pas.
